In [ ]:
import os
import re 
import json
import unicodedata
from pathlib import Path
from typing import List, Optional, Tuple, Dict
 
import pdfplumber 

In [77]:
# ------------------- Patrones -------------------
RE_CAMARA = re.compile(r"(?P<numero>\d{1,4})/(?P<anio>\d{4})C\b", re.IGNORECASE)
RE_SENADO = re.compile(r"(?P<numero>\d{1,4})/(?P<anio>\d{4})S\b", re.IGNORECASE)
RE_TIPO_LEY = re.compile(
    r"\b(PROYECTO DE LEY ORDINARIA|PROYECTO DE LEY|ACTO LEGISLATIVO(?: \w+)*)\b",
    re.IGNORECASE,
)
RE_TITULO = re.compile(
    r"\b(Por medio de la cual|Por el cual|Mediante el cual)\b",
    re.IGNORECASE,
)
INICIO_FILA = re.compile(r"^\d+\s+\d{1,4}/\d{4}C\b", re.IGNORECASE)


In [78]:
# ------------------- Funciones auxiliares -------------------

def normalize_text(s: str) -> str:
    """
    Normaliza unicode, recorta y colapsa múltiples espacios.
    """
    if not s:
        return ""
    clean = unicodedata.normalize("NFKC", s).strip()
    return re.sub(r"\s+", " ", clean)


def parse_numero(pat: re.Pattern, text: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Busca con el patrón y devuelve (numero, anio) o (None, None).
    """
    m = pat.search(text)
    if m:
        return m.group("numero"), m.group("anio")
    return None, None


def split_pseud_titulo_tipo(text: str) -> Tuple[str, str, str]:
    """
    Fallback: intenta dividir text en pseudónimo, título y tipo de ley usando regex.
    """
    # tipo de ley fallback
    m_type = re.search(
        r"\b(PROYECTO DE LEY ORDINARIA|PROYECTO DE LEY|ACTO LEGISLATIVO(?: \w+)*)\b",
        text,
        re.IGNORECASE,
    )
    tipo = m_type.group(1).strip() if m_type else ""
    pre = text[: m_type.start()].strip() if m_type else text

    # pseudónimo antes de 'Por'
    m_pseud = re.search(r"^(.+?)\bPor\b", pre, re.IGNORECASE)
    if m_pseud:
        pseud = m_pseud.group(1).strip()
        titulo = pre[m_pseud.end() - 3 :].strip()
    else:
        pseud, titulo = pre, ""
    return pseud, titulo, tipo

In [79]:
# ------------------- Procesamiento de cada bloque -------------------

def procesar_bloque_por_lineas(lineas: List[str]) -> Optional[Dict[str, str]]:
    """
    Extrae metadatos de un bloque de líneas.
    """
    # limpiar y filtrar líneas vacías
    raw = [normalize_text(l) for l in lineas if normalize_text(l)]

    # 1) identificar líneas de tipo de ley (todo uppercase al final)
    tipo_lines: List[str] = []
    for l in reversed(raw):
        if l and l.upper() == l and re.search(r"[A-Z]", l):
            tipo_lines.insert(0, l)
        else:
            break
    tipo_ley = " ".join(tipo_lines)

    # 2) contenido base sin líneas de tipo
    content = raw[: len(raw) - len(tipo_lines)] if tipo_lines else raw

    # 3) concatenar y normalizar
    base = " ".join(content)
    base = re.sub(r"\s+", " ", base).strip()

    # 4) extraer números
    num_cam, anio_cam = parse_numero(RE_CAMARA, base)
    num_sen, anio_sen = parse_numero(RE_SENADO, base)

    # 5) extraer título
    m_tit = RE_TITULO.search(base)
    titulo = base[m_tit.start() :].strip() if m_tit else ""

    # 6) extraer pseudónimo
    pseud = ""
    if num_cam:
        m_end = RE_SENADO.search(base) if num_sen else RE_CAMARA.search(base)
        end_idx = m_end.end() if m_end else 0
        start_tit = m_tit.start() if m_tit else len(base)
        pseud = base[end_idx:start_tit].strip()
        pseud = re.sub(r"^\d+\s*", "", pseud)

    # 7) fallback si falta algún campo
    if not (pseud and titulo and tipo_ley):
        fb_p, fb_t, fb_type = split_pseud_titulo_tipo(base)
        pseud = pseud or fb_p
        titulo = titulo or fb_t
        tipo_ley = tipo_ley or fb_type

    # 8) validación mínima
    if not num_cam or not pseud:
        return None

    return {
        "numeroCamara": num_cam,
        "anioCamara": anio_cam or "",
        "numeroSenado": num_sen or "",
        "anioSenado": anio_sen or "",
        "pseudonimo": pseud,
        "titulo": titulo,
        "tipoLey": tipo_ley,
    }

In [80]:
# ------------------- Lectura y guardado de PDF -------------------

def extraer_filas_desde_texto(path_pdf: str) -> List[Dict[str, str]]:
    resultados: List[Dict[str, str]] = []
    lines: List[str] = []
    with pdfplumber.open(path_pdf) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ""
            lines.extend(txt.splitlines())

    bloques, bloque_actual = [], []
    for ln in lines:
        clean = normalize_text(ln)
        if INICIO_FILA.match(clean):
            if bloque_actual:
                bloques.append(bloque_actual)
            bloque_actual = [clean]
        else:
            if bloque_actual:
                bloque_actual.append(clean)
    if bloque_actual:
        bloques.append(bloque_actual)

    for blk in bloques:
        fila = procesar_bloque_por_lineas(blk)
        if fila:
            resultados.append(fila)
    return resultados

In [81]:
def procesar_pdf_texto_individual(path_pdf: str, carpeta_salida: str) -> None:
    os.makedirs(carpeta_salida, exist_ok=True)
    base = Path(path_pdf).stem

    filas = extraer_filas_desde_texto(path_pdf)
    print(f"✅ Total de filas extraídas: {len(filas)}")

    for idx, fila in enumerate(filas, start=1):
        out = Path(carpeta_salida) / f"{base}_{idx:04d}.json"
        with out.open("w", encoding="utf-8") as f:
            json.dump(fila, f, indent=4, ensure_ascii=False)

    print(f"📁 JSON guardados en: {Path(carpeta_salida).resolve()}")

In [82]:
if __name__ == "__main__":
    # Ejemplo de uso
    pdf_entrada = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\resource\comisiones\2022 2023 LEGISLATURA_comision_1.pdf"
    carpeta_json = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\2022_2023\comision\comision_2"
    procesar_pdf_texto_individual(pdf_entrada, carpeta_json)


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


✅ Total de filas extraídas: 132
📁 JSON guardados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\2022_2023\comision\comision_2
